In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

silver_table = spark.table(
    "dbacademy.get_started_de.silver_retail"
)

# Calculate signed line value using decimal arithmetic
gold_base = (
    silver_table
    .withColumn(
        "transaction_value",
        F.col("quantity").cast("decimal(18,0)") *
        F.col("unit_price").cast(DecimalType(18, 4))
    )
    .withColumn(
        "transaction_class",
        F.when(
            F.upper(F.trim(F.col("invoice_no"))).startswith("C"),
            "cancellation"
        ).when(
            F.col("quantity") < 0,
            "negative_quantity"
        ).otherwise("positive_or_zero_quantity")
    )
    .withColumn(
        "invoice_date",
        F.to_date("invoice_timestamp")
    )
)

# Aggregate for reporting
gold_df = (
    gold_base
    .groupBy(
        "invoice_date",
        "country",
        "product_code",
        "product_description",
        "transaction_class"
    )
    .agg(
        F.count("*").alias("transaction_rows"),
        F.sum("quantity").alias("net_quantity"),
        F.sum("transaction_value").alias("net_transaction_value"),
        F.sum(
            F.when(
                F.col("quantity") > 0,
                F.col("transaction_value")
            ).otherwise(F.lit(0).cast("decimal(30,4)"))
        ).alias("positive_quantity_value"),
        F.sum(
            F.when(
                F.col("quantity") < 0,
                F.col("transaction_value")
            ).otherwise(F.lit(0).cast("decimal(30,4)"))
        ).alias("negative_quantity_value")
    )
)

display(gold_df.limit(20))

invoice_date,country,product_code,product_description,transaction_class,transaction_rows,net_quantity,net_transaction_value,positive_quantity_value,negative_quantity_value
2010-12-01,United Kingdom,85123A,WHITE HANGING HEART T-LIGHT HOLDER,positive_or_zero_quantity,17,454,1224.1800,1224.1800,0.0000
2010-12-01,United Kingdom,71053,WHITE METAL LANTERN,positive_or_zero_quantity,6,33,116.9500,116.9500,0.0000
2010-12-01,United Kingdom,84406B,CREAM CUPID HEARTS COAT HANGER,positive_or_zero_quantity,5,40,110.0000,110.0000,0.0000
2010-12-01,United Kingdom,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,positive_or_zero_quantity,9,59,213.4200,213.4200,0.0000
2010-12-01,United Kingdom,84029E,RED WOOLLY HOTTIE WHITE HEART.,positive_or_zero_quantity,14,551,1670.5500,1670.5500,0.0000
2010-12-01,United Kingdom,22752,SET 7 BABUSHKA NESTING BOXES,positive_or_zero_quantity,9,22,178.5000,178.5000,0.0000
2010-12-01,United Kingdom,21730,GLASS STAR FROSTED T-LIGHT HOLDER,positive_or_zero_quantity,5,30,127.5000,127.5000,0.0000
2010-12-01,United Kingdom,22633,HAND WARMER UNION JACK,positive_or_zero_quantity,11,181,348.6000,348.6000,0.0000
2010-12-01,United Kingdom,22632,HAND WARMER RED POLKA DOT,positive_or_zero_quantity,5,30,55.5000,55.5000,0.0000
2010-12-01,United Kingdom,84879,ASSORTED COLOUR BIRD ORNAMENT,positive_or_zero_quantity,9,224,390.5600,390.5600,0.0000


Gold Table

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbacademy.get_started_de.gold_retail_sales")

In [0]:
gold_table = spark.table(
    "dbacademy.get_started_de.gold_retail_sales"
)

print("Gold rows:", gold_table.count())

display(
    gold_table.orderBy(
        F.desc("invoice_date")
    ).limit(10)
)

Gold rows: 315986


invoice_date,country,product_code,product_description,transaction_class,transaction_rows,net_quantity,net_transaction_value,positive_quantity_value,negative_quantity_value
2011-12-09,United Kingdom,21875,KINGS CHOICE MUG,positive_or_zero_quantity,1,36,38.1600,38.1600,0.0000
2011-12-09,United Kingdom,21326,AGED GLASS SILVER T-LIGHT HOLDER,positive_or_zero_quantity,2,156,87.0000,87.0000,0.0000
2011-12-09,Norway,21481,FAWN BLUE HOT WATER BOTTLE,positive_or_zero_quantity,1,12,45.0000,45.0000,0.0000
2011-12-09,United Kingdom,23351,ROLL WRAP 50'S CHRISTMAS,positive_or_zero_quantity,4,51,96.4200,96.4200,0.0000
2011-12-09,United Kingdom,84947,ANTIQUE SILVER TEA GLASS ENGRAVED,positive_or_zero_quantity,1,24,30.0000,30.0000,0.0000
2011-12-09,United Kingdom,23338,EGG FRYING PAN RED,positive_or_zero_quantity,2,49,90.8800,90.8800,0.0000
2011-12-09,Norway,22998,TRAVEL CARD WALLET KEEP CALM,positive_or_zero_quantity,1,72,30.2400,30.2400,0.0000
2011-12-09,Norway,22727,ALARM CLOCK BAKELIKE RED,positive_or_zero_quantity,1,8,30.0000,30.0000,0.0000
2011-12-09,United Kingdom,23300,GARDENERS KNEELING PAD CUP OF TEA,positive_or_zero_quantity,3,60,109.0800,109.0800,0.0000
2011-12-09,United Kingdom,22730,ALARM CLOCK BAKELIKE IVORY,positive_or_zero_quantity,3,10,63.9100,63.9100,0.0000


Gold reconciliation test

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

silver_table = spark.table(
    "dbacademy.get_started_de.silver_retail"
)

gold_table = spark.table(
    "dbacademy.get_started_de.gold_retail_sales"
)

def calculate_totals(df, value_column):
    return df.agg(
        F.sum(
            F.col(value_column).cast(DecimalType(24, 4))
        ).alias("signed_value")
    ).first()["signed_value"]

silver_signed = calculate_totals(
    silver_table.withColumn(
        "line_value",
        F.col("quantity").cast("decimal(18,0)") *
        F.col("unit_price").cast("decimal(18,4)")
    ),
    "line_value"
)

gold_signed = calculate_totals(
    gold_table,
    "net_transaction_value"
)

print("Silver signed value:", silver_signed)
print("Gold signed value:", gold_signed)
print("Difference:", gold_signed - silver_signed)

Silver signed value: 9747747.9340
Gold signed value: 9747747.9340
Difference: 0.0000


In [0]:
gold_duplicates = (
    gold_table
    .groupBy(
        "invoice_date",
        "country",
        "product_code",
        "product_description",
        "transaction_class"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate Gold reporting keys:",
    gold_duplicates.count()
)

Duplicate Gold reporting keys: 0


Country and daily performance Dashboard

In [0]:
from pyspark.sql import functions as F

gold_table = spark.table(
    "dbacademy.get_started_de.gold_retail_sales"
)

country_daily = (
    gold_table
    .groupBy("invoice_date", "country")
    .agg(
        F.sum("net_transaction_value")
         .alias("signed_transaction_value"),

        F.sum("positive_quantity_value")
         .alias("positive_quantity_value"),

        F.sum("negative_quantity_value")
         .alias("negative_quantity_value"),

        F.sum("transaction_rows")
         .alias("transaction_rows")
    )
)

display(
    country_daily
    .orderBy(F.desc("invoice_date"))
    .limit(20)
)

invoice_date,country,signed_transaction_value,positive_quantity_value,negative_quantity_value,transaction_rows
2011-12-09,Norway,2638.6900,2638.6900,0.0000,66
2011-12-09,France,249.4500,249.4500,0.0000,15
2011-12-09,Germany,1689.7200,1689.7200,0.0000,82
2011-12-09,Belgium,208.6400,208.6400,0.0000,17
2011-12-09,United Kingdom,27345.0300,196134.1000,-168789.0700,1452
2011-12-08,USA,-269.9600,0.0000,-269.9600,11
2011-12-08,EIRE,1084.1400,1084.1400,0.0000,72
2011-12-08,Netherlands,250.6000,250.6000,0.0000,1
2011-12-08,France,146.7000,165.9000,-19.2000,6
2011-12-08,Austria,683.2000,683.2000,0.0000,7


Product Performance

In [0]:
product_performance = (
    gold_table
    .groupBy("product_code", "product_description")
    .agg(
        F.sum("net_transaction_value")
         .alias("signed_transaction_value"),

        F.sum("net_quantity")
         .alias("net_quantity"),

        F.sum("transaction_rows")
         .alias("transaction_rows")
    )
    .orderBy(F.desc("signed_transaction_value"))
)

display(product_performance.limit(20))

product_code,product_description,signed_transaction_value,net_quantity,transaction_rows
DOT,DOTCOM POSTAGE,206245.4800,1707,710
22423,REGENCY CAKESTAND 3 TIER,164762.1900,13033,2200
47566,PARTY BUNTING,98302.9800,18022,1727
85123A,WHITE HANGING HEART T-LIGHT HOLDER,97715.9900,35025,2302
85099B,JUMBO BAG RED RETROSPOT,92356.0300,47363,2159
23084,RABBIT NIGHT LIGHT,66756.5900,30680,1051
POST,POSTAGE,66230.6400,6353,1256
22086,PAPER CHAIN KIT 50'S CHRISTMAS,63791.9400,18902,1210
84879,ASSORTED COLOUR BIRD ORNAMENT,58959.7300,36381,1501
79321,CHILLI LIGHTS,53768.0600,10343,683


KPI Summary

In [0]:
from pyspark.sql import functions as F

gold_table = spark.table(
    "dbacademy.get_started_de.gold_retail_sales"
)

kpi_summary = gold_table.agg(
    F.sum("positive_quantity_value")
        .alias("positive_quantity_value"),

    F.sum("negative_quantity_value")
        .alias("negative_quantity_value"),

    F.sum("net_transaction_value")
        .alias("signed_transaction_value"),

    F.sum("transaction_rows")
        .alias("source_row_count"),

    F.countDistinct("country")
        .alias("country_count"),

    F.countDistinct("product_code")
        .alias("product_code_count")
)

display(kpi_summary)

positive_quantity_value,negative_quantity_value,signed_transaction_value,source_row_count,country_count,product_code_count
10644560.4240,-896812.4900,9747747.9340,541909,38,4070
